# YOLO11n: second approach on Penn–Fudan

Fine-tune COCO-pretrained YOLO11n on the **same prepared images and split** as the YOLOv3-tiny baseline. Run cells in order from the repository root. Python environment used here: Ultralytics 8.3.228, PyTorch 2.11.0. Install with `python -m pip install ultralytics==8.3.228 nbclient nbformat` if needed. GPU is optional; CPU execution takes longer.

The experiment reserves the 26-image test set. Only the 25-image validation set is evaluated here. Generated checkpoints, logs and dataset files are under `second_approach/runs/` and ignored by Git.

In [1]:
from pathlib import Path
import os, json, hashlib, random, time, zipfile, platform, sys, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import yaml
import torch
import ultralytics
from ultralytics import YOLO

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.git').exists()).resolve()
ROOT = REPO_ROOT / 'second_approach'
RUNS = ROOT / 'runs'
RUN_NAME = 'yolo11n_seed42_cpu' if not torch.cuda.is_available() else 'yolo11n_seed42_gpu'
RUN = RUNS / RUN_NAME
DATA_ROOT = None  # Set to the folder containing data/processed and data/splits, if desired.
EPOCHS, BATCH, IMAGE_SIZE, SEED, LR = 50, 16, 640, 42, 0.001
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RUNS.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('YOLO_CONFIG_DIR', str(RUNS / '.ultralytics'))
os.environ.setdefault('MPLCONFIGDIR', str(RUNS / '.matplotlib'))

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, allow_nan=False) + '\n')

print('device:', DEVICE, 'ultralytics:', ultralytics.__version__, 'run:', RUN)

device: 0 ultralytics: 8.3.228 run: /home/abdullloh/pedestrian-detection/second_approach/runs/yolo11n_seed42_gpu


## Data audit

Prefer a prepared dataset supplied by `DATA_ROOT`; otherwise extract `pedestrian_data.zip` from the repo root or home directory. The archive is the one used by the YOLOv3-tiny notebook. The split and labels are checked before training.

In [2]:
if DATA_ROOT is not None:
    DATA = Path(DATA_ROOT).expanduser().resolve()
elif (REPO_ROOT/'data/splits/statistics.json').exists():
    DATA = REPO_ROOT
else:
    archive = next((p for p in (REPO_ROOT/'pedestrian_data.zip', Path.home()/'pedestrian_data.zip') if p.exists()), None)
    assert archive is not None, 'Set DATA_ROOT or place pedestrian_data.zip in the repo root or home directory.'
    DATA = ROOT/'data'
    if not (DATA/'data/splits/statistics.json').exists():
        DATA.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for m in z.infolist():
                dest = (DATA/m.filename).resolve()
                assert dest.is_relative_to(DATA.resolve()) and not m.filename.startswith('/'), m.filename
                assert ((m.external_attr >> 16) & 0o170000) != 0o120000, m.filename
            z.extractall(DATA)
PROCESSED = DATA/'data/processed'
SPLITS = DATA/'data/splits'
stats = json.loads((SPLITS/'statistics.json').read_text())
manifest = {s:(SPLITS/f'{s}.txt').read_text().split() for s in ('train','val','test')}
assert [len(manifest[s]) for s in ('train','val','test')] == [119,25,26]
names = sum(manifest.values(), [])
assert len(set(names)) == 170
shuffled = sorted(names); random.Random(SEED).shuffle(shuffled)
assert set(manifest['train']) == set(shuffled[:119])
assert set(manifest['val']) == set(shuffled[119:144])
assert set(manifest['test']) == set(shuffled[144:])
assert stats['total_instances'] == 423
for split in manifest:
    image_dir, label_dir = PROCESSED/'images'/split, PROCESSED/'labels'/split
    assert {p.stem for p in image_dir.iterdir() if p.is_file()} == set(manifest[split])
    assert {p.stem for p in label_dir.glob('*.txt')} == set(manifest[split])
    boxes = 0
    for name in manifest[split]:
        rows = [list(map(float, line.split())) for line in (label_dir/f'{name}.txt').read_text().splitlines() if line.strip()]
        for row in rows:
            assert len(row) == 5 and row[0] == 0 and all(0 <= v <= 1 for v in row[1:]) and row[3] > 0 and row[4] > 0
        boxes += len(rows)
    assert boxes == stats['splits'][split]['instances']
    print(split, len(manifest[split]), 'images,', boxes, 'boxes')
DATA_YAML = RUNS/'dataset.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(PROCESSED.resolve()), 'train':'images/train', 'val':'images/val', 'test':'images/test', 'names':{0:'person'}}))
print('dataset:', DATA)

train 119 images, 290 boxes
val 25 images, 67 boxes
test 26 images, 66 boxes
dataset: /home/abdullloh/pedestrian-detection/second_approach/data


## Configuration and training

The baseline's 50 epochs, batch 16, 640 px, AdamW, LR 0.001, weight decay 0.0005, seed 42, warmup, cosine learning-rate schedule and low augmentation values are requested explicitly. `close_mosaic=0` keeps mosaic through the final epochs as in the YOLOv3 setup. Some model-specific losses and data loading behavior cannot be identical. The effective arguments are saved in `training/args.yaml`.

The initial weights can be supplied with `YOLO11_WEIGHTS=/path/to/yolo11n.pt`; otherwise the notebook checks common local paths and lets Ultralytics download the official weights. Training is not resumed silently.

In [3]:
import tempfile
weights_candidates = [os.environ.get('YOLO11_WEIGHTS'), REPO_ROOT/'yolo11n.pt', Path.home()/'p-detection/yolo11n.pt']
WEIGHTS = next((Path(x) for x in weights_candidates if x and Path(x).is_file()), Path('yolo11n.pt'))
config = dict(data=str(DATA_YAML), imgsz=IMAGE_SIZE, epochs=EPOCHS, batch=BATCH,
    optimizer='AdamW', lr0=LR, lrf=0.01, momentum=0.9, weight_decay=0.0005,
    warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.0,
    cos_lr=True, nbs=64, close_mosaic=0, seed=SEED, deterministic=True,
    amp=True, pretrained=True, device=DEVICE, workers=0, patience=0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, translate=0.1,
    scale=0.5, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.0, copy_paste=0.0,
    project=str(RUNS), name=RUN_NAME, exist_ok=False, plots=True, verbose=False)
write_json(RUNS/'requested_config.json', {'model':'YOLO11n','weights':str(WEIGHTS), 'weights_sha256':sha256(WEIGHTS) if WEIGHTS.is_file() else None, 'config':config})
assert not RUN.exists(), f'{RUN} already exists; choose a fresh RUN_NAME to retain the earlier run.'
started = time.perf_counter()
try:
    model = YOLO(str(WEIGHTS))
    model.train(**config)
finally:
    write_json(RUNS/'training_runtime.json', {'wall_seconds':time.perf_counter()-started,
        'finished_utc':datetime.now(timezone.utc).isoformat(), 'run':str(RUN)})
BEST = RUN/'weights/best.pt'
assert BEST.is_file()
history = pd.read_csv(RUN/'results.csv'); history.columns = history.columns.str.strip()
assert len(history) == EPOCHS, f'Expected {EPOCHS} epochs, found {len(history)}'
assert np.isfinite(history.select_dtypes(include='number').to_numpy()).all()
print('completed epochs:', len(history), 'training seconds:', round(time.perf_counter()-started,2))
display(history.tail(1))

New https://pypi.org/project/ultralytics/8.4.182 available 😃 Update with 'pip install -U ultralytics'


Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3768MiB)


engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/abdullloh/pedestrian-detection/second_approach/runs/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/abdullloh/p-detection/yolo11n.pt, momentum=0.9, mosaic=1.0, multi_scale=False, name=yolo11n_seed42_gpu, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=0, perspective=0.0, plots=True, pose=12.0, pr

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


...

                   all         25         67      0.965       0.94      0.979      0.866


Speed: 0.3ms preprocess, 2.8ms inference, 0.0ms loss, 0.9ms postprocess per image


Results saved to /home/abdullloh/pedestrian-detection/second_approach/runs/yolo11n_seed42_gpu


completed epochs: 50 training seconds: 161.06


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
49,50,148.734,0.56637,0.61969,0.95887,0.9694,0.94576,0.98754,0.85973,0.49415,0.49077,0.90307,0.000011,0.000011,0.000011


## Best-checkpoint validation

Evaluate validation only, with 640 px, batch 16, FP32, confidence floor 0.001, NMS IoU 0.6 and max 300 detections. Ultralytics reports mAP and maximum-F1 precision/recall. Its evaluator differs from the pinned YOLOv3 evaluator, so mAP is the primary comparison and precision/recall need that qualification.

In [4]:
BEST = RUN/'weights/best.pt'
assert BEST.exists()
VAL_DIR = RUNS/f'{RUN_NAME}_validation'
assert not VAL_DIR.exists(), 'Validation already exists; choose a fresh RUN_NAME.'
best_model = YOLO(str(BEST))
t0 = time.perf_counter()
val = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, half=False, conf=0.001, iou=0.6, max_det=300,
    project=str(RUNS), name=VAL_DIR.name, exist_ok=False, plots=True, save_txt=True, save_conf=True,
    workers=0, verbose=False)
validation_seconds = time.perf_counter()-t0
metrics = {'model':'YOLO11n', 'split':'val', 'images':25, 'precision_best_f1':float(val.box.mp),
    'recall_best_f1':float(val.box.mr), 'mAP50':float(val.box.map50), 'mAP50_95':float(val.box.map),
    'validation_wall_seconds':validation_seconds, 'preprocess_ms_per_image':float(val.speed['preprocess']),
    'inference_ms_per_image':float(val.speed['inference']), 'nms_ms_per_image':float(val.speed['postprocess']),
    'parameters':sum(p.numel() for p in best_model.model.parameters()),
    'checkpoint_sha256':sha256(BEST), 'evaluator':f'ultralytics {ultralytics.__version__} YOLO.val',
    'confidence_floor':0.001, 'nms_iou':0.6, 'max_det':300, 'inference_precision':'FP32',
    'device':str(DEVICE), 'batch_size':BATCH}
write_json(VAL_DIR/'metrics.json',metrics)
pd.DataFrame([metrics]).to_csv(VAL_DIR/'metrics.csv',index=False)
write_json(RUNS/'environment.json', {'python':sys.version, 'platform':platform.platform(),
    'torch':torch.__version__, 'ultralytics':ultralytics.__version__,
    'cuda_available':torch.cuda.is_available(), 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'device':str(DEVICE)})
print(pd.DataFrame([metrics]).to_string(index=False))

Ultralytics 8.3.228 🚀 Python-3.13.9 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 3768MiB)


YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7429.9±597.3 MB/s, size: 461.6 KB)


val: Scanning /home/abdullloh/pedestrian-detection/second_approach/data/data/processed/labels/val.cache... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25 11.7Mit/s 0.0s

val: Scanning /home/abdullloh/pedestrian-detection/second_approach/data/data/processed/labels/val.cache... 25 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 25/25 57.1Kit/s 0.0s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1s/it 2.2s

                   all         25         67      0.964       0.94      0.981      0.868


Speed: 0.4ms preprocess, 6.0ms inference, 0.0ms loss, 0.7ms postprocess per image


Results saved to /home/abdullloh/pedestrian-detection/second_approach/runs/yolo11n_seed42_gpu_validation


  model split  images  precision_best_f1  recall_best_f1    mAP50  mAP50_95  validation_wall_seconds  preprocess_ms_per_image  inference_ms_per_image  nms_ms_per_image  parameters                                                checkpoint_sha256                    evaluator  confidence_floor  nms_iou  max_det inference_precision device  batch_size
YOLO11n   val      25           0.963564        0.940299 0.980692  0.867988                  3.66218                 0.397601                6.034866          0.697009     2582347 d5aadfabcaa5b2c1409abcb9ad52b793bccd0abf3a5361c585cfb192dc28fc20 ultralytics 8.3.228 YOLO.val             0.001      0.6      300                FP32      0          16


## Report notes

| Aspect | YOLOv3-tiny baseline | YOLO11n run |
|---|---|---|
| Dataset / split | Same prepared Penn–Fudan, 119/25/26 | Same archive and seeded manifest |
| Input / epochs / batch / optimizer / LR / seed | 640 / 50 / 16 / AdamW / 0.001 / 42 | Same requested values |
| Pretraining | COCO YOLOv3-tiny | COCO YOLO11n |
| Augmentation | `hyp.scratch-low.yaml`; optional Albumentations disabled | Matching supported geometric/color settings; Albumentations availability depends on environment |
| Schedule | cosine decay, 100-iteration minimum warmup, nominal batch 64 | Explicit `cos_lr=True`, matching warmup settings, `nbs=64` |
| Mosaic | active throughout baseline | `close_mosaic=0` |
| Validation | pinned YOLOv3 evaluator | Ultralytics 8.3.228 evaluator with matched requested thresholds |
| Architecture/loss | anchor-based YOLOv3-tiny | anchor-free YOLO11n; loss terms cannot be made identical |
| Runtime | baseline's own device and package versions | recorded in `environment.json` and validation metrics |

This notebook does **not** use the held-out test images for model selection. The baseline repository notebook describes intended settings but contains no executed 50-epoch result in Git, so any quantitative comparison should use an actual saved YOLOv3 run.